# S9 - Actividad autónoma 4.1: observabilidad del pipeline del Proyecto Sello (equipo LLSW3)

**Proyecto Sello:** Sistema de monitoreo Big Data del tráfico de red del campus universitario (UPeU Juliaca).

**Fuente real:** topic `red-telemetria`. El agente `uso-red/agente/agente_red.py` lee cada segundo los contadores de la
tarjeta Wi-Fi Intel AX211 del nodo de captura (por donde sale el tráfico hoy) y los publica por MQTT; el puente los
reenvía a Kafka y el consumer `uso-red-group` los valida. Este notebook instrumenta ese pipeline: campos de
observabilidad en Spark, `lastProgress`, evidencia en Parquet, latencia por ventana, alertas y costos.

## 1. SparkSession con el conector de Kafka

In [1]:
from pyspark.sql import SparkSession
import json, os, time, urllib.request, urllib.parse

spark = (
    SparkSession.builder
    .appName("s09-proyecto-red-observabilidad")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4041")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("FATAL")
KAFKA, TOPIC = "lambda26-kafka:9092", "red-telemetria"
spark.version

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dbbecf3a-0b27-4762-b669-99cc264aea9c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central


	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 258ms :: artifacts dl 9ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central i

26/10/13 14:52:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'4.2.0'

## 2. Lectura streaming con campos de observabilidad (`isValid`, `processedAt`, `latencyMs`)

`isValid` aplica el contrato `red.lectura` de S7: campos obligatorios presentes y tasas no negativas (una tasa negativa
es un contador reiniciado, no tráfico). `latencyMs` = momento en que Spark procesa − `timestamp` que puso el agente al
leer la tarjeta: mide el recorrido agente → MQTT → puente → Kafka → Spark.

In [2]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_red = (
    StructType()
    .add("tipoEvento", StringType()).add("sensorId", StringType()).add("interfaz", StringType())
    .add("tipoInterfaz", StringType()).add("velocidadEnlaceMbps", LongType())
    .add("bytesRecibidosSeg", DoubleType()).add("bytesEnviadosSeg", DoubleType())
    .add("paquetesRecibidosSeg", DoubleType()).add("paquetesEnviadosSeg", DoubleType())
    .add("erroresEntrada", LongType()).add("erroresSalida", LongType())
    .add("descartesEntrada", LongType()).add("descartesSalida", LongType())
    .add("intervaloMs", LongType()).add("origen", StringType()).add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

observables = (
    crudo.select(col("partition"), col("offset"), col("timestamp").alias("kafkaTimestamp"),
                 from_json(col("value").cast("string"), esquema_red).alias("e"))
    .select("partition", "offset", "kafkaTimestamp", "e.*")
    .withColumn("isValid",
        col("tipoEvento").isNotNull() & col("sensorId").isNotNull() & col("timestamp").isNotNull()
        & col("velocidadEnlaceMbps").isNotNull()
        & (col("bytesRecibidosSeg") >= 0) & (col("bytesEnviadosSeg") >= 0))
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
)
observables.printSchema()

root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- interfaz: string (nullable = true)
 |-- tipoInterfaz: string (nullable = true)
 |-- velocidadEnlaceMbps: long (nullable = true)
 |-- bytesRecibidosSeg: double (nullable = true)
 |-- bytesEnviadosSeg: double (nullable = true)
 |-- paquetesRecibidosSeg: double (nullable = true)
 |-- paquetesEnviadosSeg: double (nullable = true)
 |-- erroresEntrada: long (nullable = true)
 |-- erroresSalida: long (nullable = true)
 |-- descartesEntrada: long (nullable = true)
 |-- descartesSalida: long (nullable = true)
 |-- intervaloMs: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = true)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



## 3. Consola + Parquet en paralelo, y `lastProgress`

In [3]:
CONSOLA_COLS = ["partition", "offset", "sensorId", "bytesRecibidosSeg", "bytesEnviadosSeg", "isValid", "processedAt", "latencyMs"]
ARTIFACTS = "./artifacts/red_telemetria_observabilidad"
CHECKPOINT = "./artifacts/chk_red_observabilidad"

consulta = (
    observables.select(*CONSOLA_COLS).writeStream.outputMode("append").format("console")
    .option("truncate", "false").trigger(processingTime="5 seconds").start()
)
consulta2 = (
    observables.writeStream.outputMode("append").format("parquet")
    .option("path", ARTIFACTS).option("checkpointLocation", CHECKPOINT)
    .trigger(processingTime="5 seconds").start()
)
time.sleep(30)

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+------+--------+-----------------+----------------+-------+-----------+---------+
|partition|offset|sensorId|bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt|latencyMs|
+---------+------+--------+-----------------+----------------+-------+-----------+---------+
+---------+------+--------+-----------------+----------------+-------+-----------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5046  |llsw3-wi-fi-2|1489.8           |379.2           |true   |1791903175007|1303     |
|2        |5047  |llsw3-wi-fi-2|6940.9           |6483.5          |true   |1791903175007|272      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 2
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5048  |llsw3-wi-fi-2|504.4            |1409.3          |true   |1791903180012|4250     |
|2        |5049  |llsw3-wi-fi-2|1978.3           |2129.9          |true   |1791903180012|3228     |
|2        |5050  |llsw3-wi-fi-2|1105191.1        |10082.4         |true   |1791903180012|2204     |
|2        |5051  |llsw3-wi-fi-2|763.5            |569.5           |true   |1791903180012|1182     |
|2        |5052  |llsw3-wi-fi-2|697.7            |16853.7         |true   |1791903180012|156      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 3
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5053  |llsw3-wi-fi-2|722.2            |324.1           |true   |1791903185006|4122     |
|2        |5054  |llsw3-wi-fi-2|255.8            |266.5           |true   |1791903185006|3094     |
|2        |5055  |llsw3-wi-fi-2|785753.6         |23083.4         |true   |1791903185006|2073     |
|2        |5056  |llsw3-wi-fi-2|13478.5          |51627.5         |true   |1791903185006|956      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 4
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5057  |llsw3-wi-fi-2|3192.1           |4233.3          |true   |1791903190012|4892     |
|2        |5058  |llsw3-wi-fi-2|13285.4          |7127.9          |true   |1791903190012|3828     |
|2        |5059  |llsw3-wi-fi-2|223944.4         |15283.4         |true   |1791903190012|2770     |
|2        |5060  |llsw3-wi-fi-2|782048.7         |6174.8          |true   |1791903190012|1712     |
|2        |5061  |llsw3-wi-fi-2|1673.7           |7803.5          |true   |1791903190012|543      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 5
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5062  |llsw3-wi-fi-2|3596.5           |41346.5         |true   |1791903195006|4475     |
|2        |5063  |llsw3-wi-fi-2|479.8            |426.2           |true   |1791903195006|3421     |
|2        |5064  |llsw3-wi-fi-2|280187.4         |22598.3         |true   |1791903195006|2335     |
|2        |5065  |llsw3-wi-fi-2|888037.7         |5994.4          |true   |1791903195006|1199     |
|2        |5066  |llsw3-wi-fi-2|656.6            |1702.9          |true   |1791903195006|98       |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 6
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5067  |llsw3-wi-fi-2|306.1            |255.1           |true   |1791903200008|3996     |
|2        |5068  |llsw3-wi-fi-2|405.8            |361.1           |true   |1791903200008|2832     |
|2        |5069  |llsw3-wi-fi-2|583052.3         |6644.7          |true   |1791903200008|1667     |
|2        |5070  |llsw3-wi-fi-2|1018.2           |873.7           |true   |1791903200008|609      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



In [4]:
progreso = consulta2.lastProgress
metricas = progreso["sources"][0].get("metrics", {})
print(json.dumps({
    "service": "spark-streaming", "component": "observabilidad", "topic": TOPIC,
    "batchId": progreso["batchId"], "numInputRows": progreso["numInputRows"],
    "inputRowsPerSecond": progreso["inputRowsPerSecond"],
    "processedRowsPerSecond": progreso["processedRowsPerSecond"],
    "batchDurationMs": progreso["durationMs"].get("triggerExecution"),
    "avgOffsetsBehindLatest": metricas.get("avgOffsetsBehindLatest"),
    "maxOffsetsBehindLatest": metricas.get("maxOffsetsBehindLatest"),
    "status": "idle" if progreso["numInputRows"] == 0 else "processed",
}, indent=2))

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "red-telemetria",
  "batchId": 6,
  "numInputRows": 4,
  "inputRowsPerSecond": 0.8,
  "processedRowsPerSecond": 2.0,
  "batchDurationMs": 1980,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "processed"
}


## 4. Crecimiento de la evidencia en Parquet (dos mediciones)

Mientras se mide, el nodo descarga un archivo grande por HTTP (tráfico real) para que el pipeline vea carga y no solo reposo.

In [5]:
def tamano_bytes(ruta):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(ruta) for f in fs if f.endswith(".parquet"))

def descarga_real(segundos):
    fin = time.time() + segundos
    try:
        with urllib.request.urlopen("https://proof.ovh.net/files/1Gb.dat", timeout=10) as r:
            while time.time() < fin and r.read(1 << 16):
                pass
    except Exception as ex:
        print("descarga:", ex)

time.sleep(30)
t1, filas1, bytes1 = time.time(), spark.read.parquet(ARTIFACTS).count(), tamano_bytes(ARTIFACTS)
print(f"medición 1: {filas1} filas, {bytes1/1024:.1f} KB")
descarga_real(20)
time.sleep(100)
t2, filas2, bytes2 = time.time(), spark.read.parquet(ARTIFACTS).count(), tamano_bytes(ARTIFACTS)
print(f"medición 2: {filas2} filas, {bytes2/1024:.1f} KB  ({t2 - t1:.0f} s después)")

historial = consulta2.recentProgress
consulta.stop()
consulta2.stop()
print("consultas detenidas; micro-lotes de la consulta Parquet:", len(historial))

-------------------------------------------
Batch: 7
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5071  |llsw3-wi-fi-2|397928.5         |5737.5          |true   |1791903205009|4475     |
|2        |5072  |llsw3-wi-fi-2|329475.4         |2428.9          |true   |1791903205009|3410     |
|2        |5073  |llsw3-wi-fi-2|296.9            |320.0           |true   |1791903205009|2099     |
|2        |5074  |llsw3-wi-fi-2|444.6            |230.1           |true   |1791903205009|820      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 8
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5075  |llsw3-wi-fi-2|410.0            |247.2           |true   |1791903210007|4583     |
|2        |5076  |llsw3-wi-fi-2|802050.0         |7133.5          |true   |1791903210007|3421     |
|2        |5077  |llsw3-wi-fi-2|1185.8           |960.9           |true   |1791903210007|2346     |
|2        |5078  |llsw3-wi-fi-2|740.7            |616.2           |true   |1791903210007|1100     |
|2        |5079  |llsw3-wi-fi-2|578.1            |573.9           |true   |1791903210007|16       |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 9
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5080  |llsw3-wi-fi-2|109501.8         |3974.4          |true   |1791903215015|3917     |
|2        |5081  |llsw3-wi-fi-2|308430.5         |2670.9          |true   |1791903215015|2786     |
|2        |5082  |llsw3-wi-fi-2|927.1            |5935.1          |true   |1791903215015|1586     |
|2        |5083  |llsw3-wi-fi-2|6208.8           |5205.6          |true   |1791903215015|452      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 10
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5084  |llsw3-wi-fi-2|8818.1           |14303.5         |true   |1791903220008|4372     |
|2        |5085  |llsw3-wi-fi-2|8793.8           |11054.6         |true   |1791903220008|3293     |
|2        |5086  |llsw3-wi-fi-2|603.0            |704.1           |true   |1791903220008|2229     |
|2        |5087  |llsw3-wi-fi-2|679.2            |529.4           |true   |1791903220008|1038     |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 11
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5088  |llsw3-wi-fi-2|7679.0           |21358.0         |true   |1791903225024|4969     |
|2        |5089  |llsw3-wi-fi-2|1687.4           |6234.0          |true   |1791903225024|3726     |
|2        |5090  |llsw3-wi-fi-2|5638.6           |10706.5         |true   |1791903225024|2548     |
|2        |5091  |llsw3-wi-fi-2|1321.8           |629.3           |true   |1791903225024|1494     |
|2        |5092  |llsw3-wi-fi-2|861.7            |2141.1          |true   |1791903225024|434      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



...

-------------------------------------------
Batch: 34
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5193  |llsw3-wi-fi-2|898.3            |8790.5          |true   |1791903340008|3991     |
|2        |5194  |llsw3-wi-fi-2|7861.0           |6524.9          |true   |1791903340008|2837     |
|2        |5195  |llsw3-wi-fi-2|12418.9          |52438.2         |true   |1791903340008|1768     |
|2        |5196  |llsw3-wi-fi-2|4517.4           |3998.1          |true   |1791903340008|711      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 35
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5197  |llsw3-wi-fi-2|2186.4           |1131.9          |true   |1791903345006|4570     |
|2        |5198  |llsw3-wi-fi-2|14340.5          |6388.7          |true   |1791903345006|3387     |
|2        |5199  |llsw3-wi-fi-2|134525.3         |17992.0         |true   |1791903345006|2280     |
|2        |5200  |llsw3-wi-fi-2|75402.7          |17514.7         |true   |1791903345006|1181     |
|2        |5201  |llsw3-wi-fi-2|1539.0           |3429.5          |true   |1791903345006|111      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 36
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5202  |llsw3-wi-fi-2|26867.3          |24761.6         |true   |1791903350004|4042     |
|2        |5203  |llsw3-wi-fi-2|4793.8           |1235.4          |true   |1791903350004|2969     |
|2        |5204  |llsw3-wi-fi-2|14756.3          |11446.8         |true   |1791903350004|1912     |
|2        |5205  |llsw3-wi-fi-2|2927.9           |1987.4          |true   |1791903350004|741      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



-------------------------------------------
Batch: 37
-------------------------------------------
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|partition|offset|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|isValid|processedAt  |latencyMs|
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+
|2        |5206  |llsw3-wi-fi-2|539.6            |287.9           |true   |1791903355012|4547     |
|2        |5207  |llsw3-wi-fi-2|5553.5           |3839.1          |true   |1791903355012|3345     |
|2        |5208  |llsw3-wi-fi-2|12684.3          |21071.5         |true   |1791903355012|2234     |
|2        |5209  |llsw3-wi-fi-2|1911.3           |2243.4          |true   |1791903355012|1166     |
|2        |5210  |llsw3-wi-fi-2|448.7            |390.4           |true   |1791903355012|105      |
+---------+------+-------------+-----------------+----------------+-------+-------------+---------+



medición 2: 159 filas, 253.3 KB  (122 s después)
consultas detenidas; micro-lotes de la consulta Parquet: 38


## 5. Latencia agregada por ventana de 1 minuto (sobre la evidencia persistida)

In [6]:
from pyspark.sql.functions import window, avg, min as smin, max as smax, count, round as sround, sum as ssum

guardado = spark.read.parquet(ARTIFACTS)
latencia_por_minuto = (
    guardado.groupBy(window(col("kafkaTimestamp"), "1 minute"))
    .agg(count("*").alias("numLecturas"),
         sround(avg("latencyMs"), 1).alias("avgLatencyMs"),
         smin("latencyMs").alias("minLatencyMs"), smax("latencyMs").alias("maxLatencyMs"),
         sround(avg("bytesRecibidosSeg") / 1e6, 3).alias("rxProm_MBps"),
         ssum((~col("isValid")).cast("int")).alias("invalidas"))
    .select(col("window.start").alias("inicioVentana"), col("window.end").alias("finVentana"),
            "numLecturas", "avgLatencyMs", "minLatencyMs", "maxLatencyMs", "rxProm_MBps", "invalidas")
    .orderBy("inicioVentana")
)
latencia_por_minuto.show(truncate=False)
resumen = guardado.agg(count("*").alias("lecturas"), sround(avg("latencyMs"), 1).alias("avgLatencyMs"),
                       smax("latencyMs").alias("maxLatencyMs"),
                       sround(avg(col("isValid").cast("int")) * 100, 2).alias("pctValidas")).collect()[0]
print("Resumen:", resumen.asDict())

+-------------------+-------------------+-----------+------------+------------+------------+-----------+---------+
|inicioVentana      |finVentana         |numLecturas|avgLatencyMs|minLatencyMs|maxLatencyMs|rxProm_MBps|invalidas|
+-------------------+-------------------+-----------+------------+------------+------------+-----------+---------+
|2026-10-13 14:52:00|2026-10-13 14:53:00|6          |1995.3      |156         |4250        |0.186      |0        |
|2026-10-13 14:53:00|2026-10-13 14:54:00|53         |2478.6      |17          |4967        |0.216      |0        |
|2026-10-13 14:54:00|2026-10-13 14:55:00|54         |2459.9      |6           |4983        |0.676      |0        |
|2026-10-13 14:55:00|2026-10-13 14:56:00|51         |2428.6      |44          |4905        |0.077      |0        |
+-------------------+-------------------+-----------+------------+------------+------------+-----------+---------+



Resumen: {'lecturas': 164, 'avgLatencyMs': 2439.2, 'maxLatencyMs': 4983, 'pctValidas': 100.0}


## 6. Alertas propuestas con umbrales justificados por los datos medidos

Los valores de infraestructura se consultan a Prometheus (`http://prometheus:9090`) desde el notebook.

In [7]:
def promql(expr):
    url = "http://prometheus:9090/api/v1/query?" + urllib.parse.urlencode({"query": expr})
    return [(r["metric"], float(r["value"][1])) for r in json.load(urllib.request.urlopen(url, timeout=10))["data"]["result"]]

exporter_up = promql('up{job="kafka-exporter"}')[0][1]
lag_max = max(v for _, v in promql('max_over_time(sum(kafka_consumergroup_lag{consumergroup="uso-red-group"})[15m:15s])'))
ingesta = promql('sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[2m]))')[0][1]
peor_min = latencia_por_minuto.agg(smax("avgLatencyMs")).collect()[0][0]

alertas = [
    ("Exporter caído", 'up{job="kafka-exporter"} == 0', f"up = {exporter_up:.0f}"),
    ("Agente de red sin datos", 'sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[2m])) == 0 (2 min)', f"ingesta = {ingesta:.2f} lecturas/s"),
    ("Lag alto sostenido (uso-red-group)", 'sum(kafka_consumergroup_lag{consumergroup="uso-red-group"}) > 30 (2 min)', f"máx. 15 min = {lag_max:.0f}"),
    ("Latencia alta", "avgLatencyMs > 6000 por minuto", f"peor minuto = {peor_min:.0f} ms"),
    ("Lecturas inválidas", "invalidas > 0 en un minuto", f"inválidas = {resumen['lecturas'] - round(resumen['lecturas'] * resumen['pctValidas'] / 100)}"),
]
print(f"{'Alerta':<36}{'Regla':<92}Valor medido")
for a in alertas:
    print(f"{a[0]:<36}{a[1]:<92}{a[2]}")

Alerta                              Regla                                                                                       Valor medido
Exporter caído                      up{job="kafka-exporter"} == 0                                                               up = 1
Agente de red sin datos             sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[2m])) == 0 (2 min)    ingesta = 0.92 lecturas/s
Lag alto sostenido (uso-red-group)  sum(kafka_consumergroup_lag{consumergroup="uso-red-group"}) > 30 (2 min)                    máx. 15 min = 4
Latencia alta                       avgLatencyMs > 6000 por minuto                                                              peor minuto = 2479 ms
Lecturas inválidas                  invalidas > 0 en un minuto                                                                  inválidas = 0


**Justificación.** El agente publica **1 lectura/s**: una ingesta de 0 durante 2 min solo puede significar agente o
puente caído (es el equivalente del sensor que se apaga). El *lag* del consumer de validación se mantuvo en un dígito,
así que **30** (= 30 s de lecturas sin consumir) separa el ruido de un atraso real. La latencia por minuto queda por
debajo de un *trigger* (5 s) en promedio, así que **> 6000 ms** solo salta si los eventos esperan más de un micro-lote.
Una lectura inválida es un contador reiniciado o un mensaje roto: con 100 % de válidas hoy, cualquier inválida es
señal real.

## 7. Estimación de costos y plan de escalado (con los valores medidos)

In [8]:
pico = max((p["inputRowsPerSecond"] for p in historial if p["numInputRows"] > 0), default=0)
crec_bytes_h = (bytes2 - bytes1) / (t2 - t1) * 3600
crec_filas_h = (filas2 - filas1) / (t2 - t1) * 3600
gb_mes = crec_bytes_h * 730 / 1024**3
computo = 3 * 0.0416 * 730
almacen = gb_mes * 0.023
print("Tabla 9 - pipeline red-telemetria (valores medidos)")
print(f"  Lecturas por segundo (pico inputRowsPerSecond): {pico:.2f}")
print(f"  Latencia promedio por minuto (peor minuto)     : {peor_min:.0f} ms")
print(f"  Crecimiento del Parquet                        : {crec_filas_h:,.0f} filas/h | {crec_bytes_h/1024**2:.2f} MB/h | {gb_mes:.2f} GB/mes")
print(f"  Cómputo  : 3 x t3.medium x 730 h x US$ 0.0416 = US$ {computo:.2f}/mes")
print(f"  Almacenamiento: {gb_mes:.2f} GB/mes x US$ 0.023 = US$ {almacen:.2f}/mes (se acumula cada mes sin retención)")
print(f"  TOTAL ≈ US$ {computo + almacen:.2f}/mes por nodo de captura")
print()
for nodos in (1, 10, 50):
    print(f"  {nodos:>3} nodos -> {pico*nodos:6.1f} lecturas/s, Parquet {gb_mes*nodos:7.2f} GB/mes, almacenamiento US$ {almacen*nodos:6.2f}/mes")

Tabla 9 - pipeline red-telemetria (valores medidos)
  Lecturas por segundo (pico inputRowsPerSecond): 1.20
  Latencia promedio por minuto (peor minuto)     : 2479 ms
  Crecimiento del Parquet                        : 3,217 filas/h | 4.91 MB/h | 3.50 GB/mes
  Cómputo  : 3 x t3.medium x 730 h x US$ 0.0416 = US$ 91.10/mes
  Almacenamiento: 3.50 GB/mes x US$ 0.023 = US$ 0.08/mes (se acumula cada mes sin retención)
  TOTAL ≈ US$ 91.18/mes por nodo de captura

    1 nodos ->    1.2 lecturas/s, Parquet    3.50 GB/mes, almacenamiento US$   0.08/mes
   10 nodos ->   12.0 lecturas/s, Parquet   34.97 GB/mes, almacenamiento US$   0.80/mes
   50 nodos ->   60.0 lecturas/s, Parquet  174.84 GB/mes, almacenamiento US$   4.02/mes


**Plan de escalado.** (1) Si la ingesta sube a decenas de nodos de captura (1 lectura/s cada uno), el primer cuello es
el **Parquet**: un archivo por micro-lote cada 5 s genera miles de archivos pequeños por día; se escala subiendo el
*trigger* de la consulta de persistencia a 1 min (la de alertas sigue en 5 s) y compactando. (2) Si el *lag* de
`uso-red-group` supera 30 sostenido, se agregan particiones a `red-telemetria` y consumidores al grupo (hoy 1 partición
recibe todo porque hay una sola key). (3) El cómputo (3 instancias) domina el costo; el almacenamiento solo pesa si se
retiene para siempre, así que se propone retención de 30 días en crudo y agregados por minuto a largo plazo.